# GPT-2 Smoke Test

This notebook installs the required dependencies, checks GPU access, sets up the project folder structure, loads GPT-2, and runs a forward pass smoke test.

## 1. Install Dependencies

Install `transformers`, `tokenizers`, `datasets`, `peft`, `torch`, and supporting libraries in Google Colab.


In [ ]:
!pip install -q --upgrade transformers tokenizers datasets peft torch accelerate safetensors


## 2. Verify GPU Access

Confirm GPU availability in Colab and verify that PyTorch can use CUDA.


In [ ]:
import torch
print('torch version:', torch.__version__)

# Check for MPS (Apple Silicon GPU) first
if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    print('MPS available:', torch.backends.mps.is_available())
    device = 'mps'
    print('Using device: mps')
elif torch.cuda.is_available():
    print('CUDA available:', torch.cuda.is_available())
    device = 'cuda'
    print('CUDA device:', torch.cuda.get_device_name(0))
    # Run nvidia-smi if CUDA is available
    !nvidia-smi
else:
    print('No GPU available, using CPU')
    device = 'cpu'

## 3. Set Project Root

Set the project root directory (parent of this notebooks/ folder).


In [ ]:
import os

# Robustly find project root by walking up from CWD until README.md is found.
# This works whether the notebook is run locally (Jupyter) or in Colab,
# and regardless of which directory the kernel starts in.
_candidate = os.path.abspath(os.getcwd())
while True:
    if os.path.exists(os.path.join(_candidate, 'README.md')):
        base_dir = _candidate
        break
    _parent = os.path.dirname(_candidate)
    if _parent == _candidate:  # reached filesystem root
        base_dir = os.path.abspath(os.getcwd())  # fallback
        break
    _candidate = _parent

print('Project root:', base_dir)


## 4. Create Repo Folder Structure

Ensure the project directory structure exists: `data/raw`, `data/processed`, `tokenizers`, `models`, `scripts`, `report`, `configs`, `demo`.


In [ ]:
import os

# Canonical folder structure (matches README)
repo_dirs = ['data/raw', 'data/processed', 'tokenizers', 'models', 'scripts', 'report', 'configs', 'demo']
for d in repo_dirs:
    full_path = os.path.join(base_dir, d)
    os.makedirs(full_path, exist_ok=True)
    print('Created or exists:', full_path)

print('Folder structure ready.')


## 5. Load GPT-2 and Tokenizer

Load the GPT-2 model and tokenizer, then move the model to the available device.


In [ ]:
from transformers import GPT2TokenizerFast, GPT2LMHeadModel

# device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', device)

tokenizer = GPT2TokenizerFast.from_pretrained('gpt2')
model = GPT2LMHeadModel.from_pretrained('gpt2')
model.to(device)
print('Loaded GPT-2 and moved to device.')


## 6. Run GPT-2 Forward Pass Smoke Test

Tokenize a sample prompt, run the model, and verify that logits are produced.


In [ ]:
prompt = 'Hello, this is a quick GPT-2 smoke test.'
inputs = tokenizer(prompt, return_tensors='pt')
inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.no_grad():
    outputs = model(**inputs)
logits = outputs.logits
print('Logits shape:', logits.shape)
print('Smoke test passed: logits produced successfully.')


## 7. Save Test Outputs and Verify Paths

Optionally save a small artifact or log to the repository folders and verify the notebook can access the expected paths.


In [ ]:
import json

artifact_path = os.path.join(base_dir, 'report', 'gpt2_smoke_test_result.json')
result = {
    'prompt': prompt,
    'device': device,
    'logits_shape': list(logits.shape),
    'repo_dirs': repo_dirs
}
with open(artifact_path, 'w') as f:
    json.dump(result, f, indent=2)

print('Saved test result to:', artifact_path)
print('Repo directories:')
for d in repo_dirs:
    print(' -', os.path.join(base_dir, d))
